In [ ]:
import os
from functools import reduce
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    GridSearchCV, 
    train_test_split, 
    KFold
)
from sklearn.preprocessing import (
    MinMaxScaler, 
    StandardScaler, 
    normalize
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    silhouette_score
)

from sklearn.ensemble import RandomForestRegressor
from skopt import gp_minimize, BayesSearchCV
from skopt.space import Real, Integer
from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import (
    AllChem,
    MACCSkeys,
    ChemicalFeatures,
    Lipinski,
    rdmolops,
    MolStandardize
)
param_space = {
    'n_estimators': Integer(50, 200),
    'max_depth': Integer(3, 10), 
    'max_features': ['sqrt', 'log2']
}
def get_fps(smi_list):
    fps = []
    estate = []
    for i in range(len(smi_list)):
        mol = Chem.MolFromSmiles(MolStandardize.standardize_smiles(smi_list[i]))
        fps2 = AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048)
        fps.append(fps2)
    fps = np.array(fps)
    return fps
for target_name in ['ESOL','bace','FreeSolv','FreeSolv_cal','ro','Lipophilicity', 'qm8']:
    for noise_portion in [0.1,0.3,0.5,0.7,0.9]:
        for sigma in [1,2,3]:
            for seed in []:
                path = os.getcwd()
                data = pd.read_csv(f'{path}/{target_name}/{sigma}_{noise_portion}/noise/{target_name}_train_{noise_portion}_{sigma}_{seed}.csv').iloc[:,:2]
                data.columns = ['SMILES','Label']
                origin_smiles = data['SMILES'].tolist()
                origin_labels = data['Label'].values
                origin_fingerprints = get_fps(origin_smiles)


                scaler = StandardScaler().fit(origin_labels.reshape(-1,1))
                normed_labels = scaler.transform(origin_labels.reshape(-1,1))

                opt = BayesSearchCV(
                    estimator=RandomForestRegressor(random_state=100, n_jobs=60),
                    search_spaces=param_space,
                    n_iter=8,
                    n_jobs=60,
                    scoring='neg_mean_absolute_error',
                    random_state=42,
                    verbose=0
                )

                opt.fit(origin_fingerprints,normed_labels.ravel())
                
                best_params = opt.best_params_
                reg = RandomForestRegressor(
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    max_features=best_params['max_features'],
                    n_jobs=60,
                    random_state=100
                )
                
                reg = reg.fit(origin_fingerprints,normed_labels.ravel())
                train_predictions = scaler.inverse_transform(reg.predict(origin_fingerprints).reshape(-1,1))
                train_bias = origin_labels.reshape(-1,1) -train_predictions.reshape(-1,1)
                uncertainty_predictions = np.zeros((origin_fingerprints.shape[0], len(reg.estimators_)))
                for i, tree in enumerate(reg.estimators_):
                    tree_pred = tree.predict(origin_fingerprints)
                    uncertainty_predictions[:, i] = tree_pred
                train_uncertainty = np.std(uncertainty_predictions,axis=1).reshape(-1,1)
                train_bias = np.concatenate([np.array(origin_smiles).reshape(-1,1),train_bias],axis=0)
                train_uncertainty = np.concatenate([np.array(origin_smiles).reshape(-1,1),train_uncertainty],axis=0)
                pd.DataFrame(train_bias).to_csv(f'{path}/{target_name}/train_valid_test/{sigma}_{noise_portion}/noise/{target_name}_train_{noise_portion}_{sigma}_{seed}_bias_for_cor.csv',index=False)
                pd.DataFrame(train_uncertainty).to_csv(f'{path}/{target_name}/train_valid_test/{sigma}_{noise_portion}/noise/{target_name}_train_{noise_portion}_{sigma}_{seed}_uncert_for_cor.csv',index=False)

The script was used to produce the bias and uncertainty for unveiling the noise-response behaviors